# Проверка: договоры торгового эквайринга, май–июль 2026

Скрипт: `sources/sql/trade_acquiring_actual_clients.sql` (задача 12554).

Зерно — **договор** (`n_agr`), как в прогнозе коллег. Уникальный ИНН в сверку числа не входит.

Прогноз, договоры: май **6468**, июнь **6685**, июль **6789**.

Периметр — секция 01 `final_df`. Договор входит в месяц, если `d_valid_from <= month_end` и (`d_valid_to` пустая или `>= month_start`), и есть неудалённая привязка `agr_terms` с `cf_ter_type = 'P'`, которая пересекает месяц: `d_valid_from <= month_end` и (`d_valid_to` пустая или `> month_start`).

Колонки строки договора: ИНН, ЦФТ ID, номер договора, дата открытия, дата закрытия, РФ.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.0f}'.replace(',', ' '))

LAKE_USER = 'Shestopalov-VYur'
LAKE_PASSWORD = None
for cand in [
    Path.cwd(),
    Path.cwd() / 'sources' / 'sql',
    Path('/home/jovyan/documents/Equaring'),
    Path('/home/jovyan/documents/Equaring/sources/sql'),
]:
    if (cand / 'connection_secrets.py').exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        from connection_secrets import LAKE_USER, LAKE_PASSWORD
        break

user_params = {'user_name': LAKE_USER}
if LAKE_PASSWORD:
    user_params['password'] = LAKE_PASSWORD

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params=user_params,
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
MONTHS = [
    ('2026-05', '2026-05-01', '2026-05-31'),
    ('2026-06', '2026-06-01', '2026-06-30'),
    ('2026-07', '2026-07-01', '2026-07-31'),
]
MEM_LIMIT = '16g'

months_sql = '\n  union all\n'.join(
    "  select '{label}' as report_month, cast('{month_start}' as date) as month_start, cast('{month_end}' as date) as month_end".format(
        label=label, month_start=month_start, month_end=month_end,
    )
    for label, month_start, month_end in MONTHS
)

sql_counts = f"""
with months as (
{months_sql}
),
sa_open as (
  select
    m.report_month,
    cast(a.n_agr as string) as n_agr,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
  from months m
  join ods_alpha.scd1_agreements a
    on cast(a.d_valid_from as date) <= m.month_end
   and (a.d_valid_to is null or cast(a.d_valid_to as date) >= m.month_start)
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
),
terms_p as (
  select distinct
    m.report_month,
    cast(t.n_agr as string) as n_agr
  from months m
  join ods_alpha.scd1_agr_terms t
    on cast(t.d_valid_from as date) <= m.month_end
   and (t.d_valid_to is null or cast(t.d_valid_to as date) > m.month_start)
  where upper(trim(cast(t.cf_ter_type as string))) = 'P'
    and coalesce(t.ods_deleted_flg, '0') <> '1'
)
select
  s.report_month,
  count(distinct case when s.inn <> '' then s.inn end) as clients_sa_open,
  count(distinct s.n_agr) as contracts_sa_open,
  count(distinct case when tp.n_agr is not null and s.inn <> '' then s.inn end) as clients_with_p,
  count(distinct case when tp.n_agr is not null then s.n_agr end) as contracts_with_p
from sa_open s
left join terms_p tp
  on tp.report_month = s.report_month
 and tp.n_agr = s.n_agr
group by s.report_month
order by s.report_month
"""

with imp:
    imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
    counts = imp.fetch(sql_counts)

counts = counts.copy()
for col in ['clients_sa_open', 'contracts_sa_open', 'clients_with_p', 'contracts_with_p']:
    counts[col] = pd.to_numeric(counts[col], errors='coerce').fillna(0).astype('int64')

counts['clients_dropped_by_p'] = counts['clients_sa_open'] - counts['clients_with_p']
counts['contracts_dropped_by_p'] = counts['contracts_sa_open'] - counts['contracts_with_p']

FORECAST_CONTRACTS = {
    '2026-05': 6468,
    '2026-06': 6685,
    '2026-07': 6789,
}

show = pd.DataFrame({
    'Месяц': counts['report_month'].astype(str),
    'Договоры': counts['contracts_with_p'],
})
show['Прогноз коллег'] = show['Месяц'].map(FORECAST_CONTRACTS)
show['Разница'] = show['Договоры'] - show['Прогноз коллег']
show['Договоры SA без фильтра P'] = counts['contracts_sa_open'].to_numpy()
show['Договоры отсечены фильтром P'] = counts['contracts_dropped_by_p'].to_numpy()
show


## Таблица договора

Одна строка — один договор месяца. Число строк за месяц сверяется с прогнозом коллег.

Колонки: ИНН, ЦФТ ID, номер договора, дата открытия, дата закрытия, РФ. ЦФТ и РФ подтягиваются той же цепочкой, что секции 03 и 06 `final_df`. Месяцы считаются по очереди.


In [ ]:
PRODUCT_COLUMNS = [
    'Месяц',
    'ИНН',
    'ЦФТ ID',
    'Номер договора',
    'Дата открытия договора',
    'Дата закрытия договора',
    'РФ, в котором обслуживается клиент',
]


def sql_product(month_start, month_end):
    return f"""
WITH sa_contracts AS (
  SELECT DISTINCT
    cast(a.n_agr AS string) AS n_agr,
    trim(cast(a.c_agr_number AS string)) AS contract_number,
    cast(a.d_valid_from AS date) AS open_dt,
    cast(a.d_valid_to AS date) AS close_dt,
    regexp_replace(trim(cast(c.c_inn AS string)), '[^0-9]', '') AS inn
  FROM ods_alpha.scd1_agreements a
  JOIN ods_alpha.scd1_companies c
    ON c.n_cmp = a.n_cmp_client
  WHERE upper(trim(cast(a.acq_class AS string))) = 'SA'
    AND cast(a.d_valid_from AS date) <= cast('{month_end}' as date)
    AND (
      a.d_valid_to IS NULL
      OR cast(a.d_valid_to AS date) >= cast('{month_start}' as date)
    )
    AND coalesce(a.ods_deleted_flg, '0') <> '1'
    AND coalesce(c.ods_deleted_flg, '0') <> '1'
    AND c.c_inn IS NOT NULL
    AND EXISTS (
      SELECT 1
      FROM ods_alpha.scd1_agr_terms t
      WHERE cast(t.n_agr AS string) = cast(a.n_agr AS string)
        AND cast(t.d_valid_from AS date) <= cast('{month_end}' as date)
        AND (
          t.d_valid_to IS NULL
          OR cast(t.d_valid_to AS date) > cast('{month_start}' as date)
        )
        AND upper(trim(cast(t.cf_ter_type AS string))) = 'P'
        AND coalesce(t.ods_deleted_flg, '0') <> '1'
    )
),

sa_inns AS (
  SELECT DISTINCT inn
  FROM sa_contracts
  WHERE inn IS NOT NULL
    AND inn <> ''
),

ocrm_ranked AS (
  SELECT
    i.inn,
    cast(soe.row_id AS string) AS row_id,
    row_number() OVER (
      PARTITION BY i.inn
      ORDER BY cast(soe.created AS timestamp) DESC, cast(soe.row_id AS string) DESC
    ) AS rn
  FROM sa_inns i
  JOIN ocrm_ul.s_org_ext soe
    ON regexp_replace(trim(cast(soe.x_inn AS string)), '[^0-9]', '') = i.inn
  WHERE coalesce(soe.x_removed_flg, 'N') = 'N'
    AND coalesce(soe.x_duplicate_flg, 'N') = 'N'
),

ocrm_one AS (
  SELECT inn, row_id
  FROM ocrm_ranked
  WHERE rn = 1
),

cdi_ranked AS (
  SELECT
    o.inn,
    cast(e.party_id AS string) AS cdi_id,
    row_number() OVER (
      PARTITION BY o.inn
      ORDER BY
        CASE WHEN e.party_id IS NULL THEN 1 ELSE 0 END,
        cast(e.party_id AS string) DESC
    ) AS rn
  FROM ocrm_one o
  LEFT JOIN cdiul.ext_id_org e
    ON cast(e.cmo_ext_party_source_id AS string) = o.row_id
   AND upper(cast(e.cmo_ext_source_system AS string)) LIKE 'OCRM%'
),

cdi_one AS (
  SELECT inn, cdi_id
  FROM cdi_ranked
  WHERE rn = 1
    AND cdi_id IS NOT NULL
    AND cdi_id <> ''
),

cft_links AS (
  SELECT
    c.inn,
    c.cdi_id,
    cast(e.cmo_ext_party_source_id AS string) AS cft_id
  FROM cdi_one c
  JOIN cdiul.ext_id_org e
    ON cast(e.party_id AS string) = c.cdi_id
   AND upper(cast(e.cmo_ext_source_system AS string)) LIKE 'CFT%'
  WHERE e.cmo_ext_party_source_id IS NOT NULL
),

cft_ids AS (
  SELECT DISTINCT cft_id
  FROM cft_links
  WHERE cft_id IS NOT NULL
    AND cft_id <> ''
),

r2_ranked AS (
  SELECT
    cast(m.c_cl_org AS string) AS cft_id,
    m.c_depart AS c_depart_raw,
    row_number() OVER (
      PARTITION BY cast(m.c_cl_org AS string)
      ORDER BY
        CASE WHEN m.c_tariff_plan IS NOT NULL THEN 0 ELSE 1 END,
        CASE WHEN m.c_depart IS NOT NULL THEN 0 ELSE 1 END,
        m.id DESC
    ) AS rn
  FROM ods.scd1_z_r2_ip_merchants m
  JOIN cft_ids ids
    ON cast(m.c_cl_org AS string) = ids.cft_id
  WHERE m.c_cl_org IS NOT NULL
    AND coalesce(m.ods_deleted_flg, '0') <> '1'
),

r2_one AS (
  SELECT
    r2.cft_id,
    nullif(trim(cast(br.c_shortlabel AS string)), '') AS filial_rf_raw
  FROM r2_ranked r2
  LEFT JOIN ods.scd1_z_depart dep
    ON dep.id = r2.c_depart_raw
  LEFT JOIN ods.scd1_z_branch br
    ON br.id = dep.c_filial
  WHERE r2.rn = 1
),

contract_cft AS (
  SELECT
    s.n_agr,
    s.contract_number,
    s.open_dt,
    s.close_dt,
    s.inn,
    l.cft_id,
    r.filial_rf_raw,
    row_number() OVER (
      PARTITION BY s.n_agr
      ORDER BY
        CASE WHEN r.filial_rf_raw IS NOT NULL THEN 0 ELSE 1 END,
        CASE WHEN l.cft_id IS NOT NULL AND l.cft_id <> '' THEN 0 ELSE 1 END,
        l.cft_id
    ) AS rn
  FROM sa_contracts s
  LEFT JOIN cft_links l
    ON l.inn = s.inn
  LEFT JOIN r2_one r
    ON r.cft_id = l.cft_id
  WHERE s.inn IS NOT NULL
    AND s.inn <> ''
),

filial_norm AS (
  SELECT
    n_agr,
    contract_number,
    open_dt,
    close_dt,
    inn,
    cft_id,
    filial_rf_raw,
    regexp_replace(
      CASE
        WHEN instr(lower(filial_rf_raw), 'рф') > 0
          THEN substr(filial_rf_raw, 1, instr(lower(filial_rf_raw), 'рф') + 1)
        ELSE filial_rf_raw
      END,
      '[[:space:]]+',
      ' '
    ) AS filial_cut
  FROM contract_cft
  WHERE rn = 1
),

filial_clean AS (
  SELECT
    n_agr,
    contract_number,
    open_dt,
    close_dt,
    inn,
    cft_id,
    nullif(
      regexp_replace(
        regexp_replace(
          lower(trim(filial_cut)),
          '\\s+(ао|пао|оао|зао)\\b.*$',
          ''
        ),
        '[ ,;.-]+$',
        ''
      ),
      ''
    ) AS filial_low
  FROM filial_norm
)

SELECT
  inn AS inn,
  cft_id AS cft_id,
  nullif(contract_number, '') AS contract_number,
  open_dt AS open_dt,
  close_dt AS close_dt,
  CASE
    WHEN filial_low IS NULL THEN NULL
    ELSE regexp_replace(
      concat(upper(substr(filial_low, 1, 1)), substr(filial_low, 2)),
      'рф',
      'РФ'
    )
  END AS filial_rf
FROM filial_clean
"""


frames = []
with imp:
    imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
    for label, month_start, month_end in MONTHS:
        print(label, flush=True)
        part = imp.fetch(sql_product(month_start, month_end))
        if part is None or part.empty:
            part = pd.DataFrame(columns=['inn', 'cft_id', 'contract_number', 'open_dt', 'close_dt', 'filial_rf'])
        part = part.copy()
        part.insert(0, 'report_month', label)
        frames.append(part)
        print(label, 'contracts', len(part), flush=True)

product_df = pd.concat(frames, ignore_index=True)
product_df = product_df.rename(columns={
    'report_month': 'Месяц',
    'inn': 'ИНН',
    'cft_id': 'ЦФТ ID',
    'contract_number': 'Номер договора',
    'open_dt': 'Дата открытия договора',
    'close_dt': 'Дата закрытия договора',
    'filial_rf': 'РФ, в котором обслуживается клиент',
})
product_df = product_df[PRODUCT_COLUMNS]

by_month = (
    product_df.groupby('Месяц', as_index=False)
    .agg(
        Договоры=('Номер договора', 'size'),
        ЦФТ_заполнено=('ЦФТ ID', lambda s: int(s.notna().sum())),
        РФ_заполнено=('РФ, в котором обслуживается клиент', lambda s: int(s.notna().sum())),
    )
)
by_month['Прогноз коллег'] = by_month['Месяц'].map(FORECAST_CONTRACTS)
by_month['Разница'] = by_month['Договоры'] - by_month['Прогноз коллег']
by_month['Доля ЦФТ'] = (by_month['ЦФТ_заполнено'] / by_month['Договоры']).round(3)
by_month['Доля РФ'] = (by_month['РФ_заполнено'] / by_month['Договоры']).round(3)

print('строк всего', len(product_df))
display(by_month)
product_df.head(20)


## Какие коды `cf_ter_type` есть у договоров SA за июль 2026

Те же границы месяца, что в секции 01: договор пересекает июль, привязка пересекает июль (`d_valid_to > 2026-07-01`). `P` остаётся в периметре. Остальные коды нет.


In [ ]:
MONTH_START = '2026-07-01'
MONTH_END = '2026-07-31'

sql_types = f"""
with sa_open as (
  select cast(a.n_agr as string) as n_agr
  from ods_alpha.scd1_agreements a
  join ods_alpha.scd1_companies c
    on c.n_cmp = a.n_cmp_client
  where upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{MONTH_END}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{MONTH_START}' as date))
    and coalesce(a.ods_deleted_flg, '0') <> '1'
    and coalesce(c.ods_deleted_flg, '0') <> '1'
    and c.c_inn is not null
)
select
  coalesce(nullif(upper(trim(cast(t.cf_ter_type as string))), ''), '(пусто)') as cf_ter_type,
  count(distinct cast(t.n_agr as string)) as contracts,
  count(*) as term_rows
from ods_alpha.scd1_agr_terms t
join sa_open s
  on s.n_agr = cast(t.n_agr as string)
where cast(t.d_valid_from as date) <= cast('{MONTH_END}' as date)
  and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{MONTH_START}' as date))
  and coalesce(t.ods_deleted_flg, '0') <> '1'
group by 1
order by contracts desc
"""

with imp:
    imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
    types = imp.fetch(sql_types)

types = types.copy()
types['contracts'] = pd.to_numeric(types['contracts'], errors='coerce').fillna(0).astype('int64')
types['term_rows'] = pd.to_numeric(types['term_rows'], errors='coerce').fillna(0).astype('int64')
types['in_script'] = types['cf_ter_type'].eq('P')
types
